# Microphone and simulation observer locations

The workbook contains 24 measurement microphones and five simulation observers. Both panels use an x–z plane projection: 0° points along +z and angles increase toward +x. Polar radius is `sqrt(x² + z²)` in workbook coordinate units; the workbook's `y` coordinate is not shown. Angles are derived from `x` and `z` so the two groups share one convention. Microphones are numbered in worksheet row order.

In [ ]:
from pathlib import Path
import numpy as np
from openpyxl import load_workbook

# Works when the kernel starts in this notebook's directory or the repository root.
workbook_candidates = (
    Path('Mics_MHR.xlsx'),
    Path('DUC/results_postprocessing/POC2/Mics_MHR.xlsx'),
    Path('C:/git/flow360cases/DUC/results_postprocessing/POC2/Mics_MHR.xlsx'),
)
workbook_path = next((path for path in workbook_candidates if path.is_file()), None)
if workbook_path is None:
    raise FileNotFoundError('Set workbook_candidates to the location of Mics_MHR.xlsx')

workbook = load_workbook(workbook_path, read_only=True, data_only=True)
try:
    sheet = workbook.active
    rows = list(sheet.iter_rows(values_only=True))
finally:
    workbook.close()

if tuple(rows[1][:5]) != ('x', 'y', 'z', 'r', 'theta') or tuple(rows[1][6:11]) != ('x', 'y', 'z', 'r', 'theta'):
    raise ValueError('Unexpected coordinate columns in Mics_MHR.xlsx')

def has_xyz(values):
    return all(isinstance(value, (int, float)) and np.isfinite(value) for value in values)

measurement_xyz = np.asarray(
    [row[:3] for row in rows[2:] if has_xyz(row[:3])], dtype=float
)
simulation_xyz = np.asarray(
    [row[6:9] for row in rows[2:] if has_xyz(row[6:9])], dtype=float
)
if not len(measurement_xyz) or not len(simulation_xyz):
    raise ValueError('No complete microphone or simulation observer coordinates found')

def polar_xz(xyz):
    x, z = xyz[:, 0], xyz[:, 2]
    return np.mod(np.arctan2(x, z), 2 * np.pi), np.hypot(x, z)

mic_theta, mic_radius = polar_xz(measurement_xyz)
sim_theta, sim_radius = polar_xz(simulation_xyz)
print(f'Loaded {len(mic_theta)} measurement microphones and {len(sim_theta)} simulation observers from {workbook_path.name}')
print(f'Microphone x-z radius: {np.mean(mic_radius):.2f}; simulation observer x-z radius: {np.mean(sim_radius):.2f} (workbook units)')

In [ ]:
import matplotlib.pyplot as plt

mic_color = '#087E8B'
observer_color = '#D95D39'
text_color = '#263746'
fig, (ax_full, ax_zoom) = plt.subplots(
    1, 2, subplot_kw={'projection': 'polar'},
    figsize=(15, 7), gridspec_kw={'width_ratios': [1.1, 1]}
)
fig.patch.set_facecolor('white')
angle_ticks = np.arange(0, 360, 30)
angle_labels = [f'{angle}°' for angle in angle_ticks]
for angle, label in ((0, '+z'), (90, '+x'), (180, '-z'), (270, '-x')):
    angle_labels[angle // 30] = f'{angle}° ({label})'
for axis in (ax_full, ax_zoom):
    axis.set_theta_zero_location('N')
    axis.set_theta_direction(-1)
    axis.set_thetagrids(angle_ticks, angle_labels)
    axis.set_facecolor('#FAFCFD')
    axis.tick_params(colors=text_color, labelsize=9)
    axis.grid(color='#A8B7C3', alpha=0.55)
    axis.spines['polar'].set_color('#8CA2B2')
    axis.set_rlabel_position(225)

closed_theta = np.r_[mic_theta, mic_theta[0]]
closed_radius = np.r_[mic_radius, mic_radius[0]]
ax_full.set_ylim(0, 112)
ax_full.set_rticks([15, 50, 100])
ax_full.plot(closed_theta, closed_radius, color=mic_color, linewidth=1.4, alpha=0.7)
ax_full.scatter(mic_theta, mic_radius, s=35, color=mic_color, zorder=3, label='Measurement microphones')
ax_full.scatter(sim_theta, sim_radius, s=100, color=observer_color, marker='D',
                edgecolor='white', linewidth=1, zorder=4, label='Simulation observers')
for number, (theta, radius) in enumerate(zip(sim_theta, sim_radius), start=1):
    ax_full.text(theta, radius - 9, f'O{number}', ha='center', va='center',
                 fontsize=10, fontweight='bold', color='#933B20')
ax_full.set_title('Full x–z layout', pad=24, fontsize=13,
                  fontweight='bold', color=text_color)

ax_zoom.set_ylim(0, 19.5)
ax_zoom.set_rticks([5, 10, 15])
ax_zoom.plot(closed_theta, closed_radius, color=mic_color, linewidth=1.5, alpha=0.7)
ax_zoom.scatter(mic_theta, mic_radius, s=55, color=mic_color,
                edgecolor='white', linewidth=0.7, zorder=3)
for number, theta in enumerate(mic_theta, start=1):
    ax_zoom.text(theta, 17.1, str(number), ha='center', va='center',
                 fontsize=8.5, color='#075762')
ax_zoom.set_title('Measurement ring · microphones 1–24', pad=24,
                  fontsize=13, fontweight='bold', color=text_color)

fig.suptitle('MHR microphone locations', fontsize=18, fontweight='bold',
             color=text_color, y=0.99)
fig.text(0.5, 0.925,
         'x–z radius in workbook units: microphones ≈15, observers ≈100 · 0° at +z, clockwise toward +x',
         ha='center', fontsize=10, color='#52636F')
legend = fig.legend(*ax_full.get_legend_handles_labels(), loc='lower center',
                    bbox_to_anchor=(0.5, 0.015), ncol=2, frameon=False)
for label in legend.get_texts():
    label.set_color(text_color)
fig.subplots_adjust(left=0.04, right=0.96, top=0.82, bottom=0.15, wspace=0.25)
plt.show()